# 2.4. Perform nested regression with bootstrapping of metric value against magnitude of ablation and biological covariate confluence

Assesses how each metrics is able to capture the variation in ablation magnitude while
being unbiased across biological covariates.

In [1]:
import pathlib
from typing import Optional

import pandas as pd
import polars as pl

from image_ablation_analysis.regression.nested_regression import (
    bootstrap_nested_regression,
    BootstrapConfig,
    ColumnSpec,
)

## Pathing

In [2]:
results_dir = pathlib.Path(".") / "results"
if not results_dir.exists():
    raise FileNotFoundError(f"Results directory not found at {results_dir.resolve()}")

regression_input_data_file = results_dir / "for_regression_subsampled.parquet"
if not regression_input_data_file.exists():
    raise FileNotFoundError(f"Regression input data not found at {regression_input_data_file.resolve()}")

## Regression helper

In [3]:
def summarize_r2_scatter_bootstrap(
    boot_df: pd.DataFrame,
    output_csv: Optional[str | pathlib.Path] = None,
    group_cols: tuple[str, ...] = ("metric_name", "ablation_type"),
    restricted_col: str = "r2_restricted",
    partial_col: str = "partial_r2_x2",
    ci: float = 0.95,
) -> pd.DataFrame:
    
    required = set(group_cols) | {"boot_idx", restricted_col, partial_col}
    missing = sorted(required - set(boot_df.columns))
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    lower_q = (1 - ci) / 2
    upper_q = 1 - lower_q

    df = boot_df.copy()
    df[restricted_col] = pd.to_numeric(df[restricted_col], errors="coerce")
    df[partial_col] = pd.to_numeric(df[partial_col], errors="coerce")

    summary = (
        df.groupby(list(group_cols), dropna=False)
        .agg(
            n_boot=("boot_idx", "nunique"),

            restricted_r2_mean=(restricted_col, "mean"),
            restricted_r2_lower=(restricted_col, lambda x: x.quantile(lower_q)),
            restricted_r2_upper=(restricted_col, lambda x: x.quantile(upper_q)),

            partial_r2_mean=(partial_col, "mean"),
            partial_r2_lower=(partial_col, lambda x: x.quantile(lower_q)),
            partial_r2_upper=(partial_col, lambda x: x.quantile(upper_q)),
        )
        .reset_index()
        .sort_values(list(group_cols))
        .reset_index(drop=True)
    )

    if output_csv is not None:
        output_csv = pathlib.Path(output_csv)
        output_csv.parent.mkdir(parents=True, exist_ok=True)
        summary.to_csv(output_csv, index=False)

    return summary

In [4]:
regression_input = pl.read_parquet(regression_input_data_file).to_pandas()
print(len(regression_input))
regression_input.head()

1844400


,created_at,run_id,original_abs_path,original_rel_path,aug_abs_path,aug_rel_path,variant,config_id,params_json,param_fixed,...,Metadata_PositionX,Metadata_PositionY,Metadata_PositionZ,Metadata_Row,Metadata_Reimaged,ablation_package,ablation_type,hash,metric_name,metric_value
0,20260218T212130127579Z,0f57f65f-729e-4cc4-a651-121c34b36181,/mnt/data_nvme1/data/ALSF_pilot_data/SN0313537...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,/mnt/hdd20tb/alsf_ablation/SN0313537/BR0014397...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,xform_abl_gaussnoise=0.2_6339a071e1e884db,albumentations:GaussNoise:6339a071e1e884db,"{""backend"":""albumentations"",""transform_name"":""...",[],...,-0.000646,0.000646,-0.000006,3.0,False,albumentations,GaussNoise,6339a071e1e884db,lpips,0.712087
1,20260219T025541654122Z,41811ddb-a68d-4909-830b-cee3d15b03ee,/mnt/data_nvme1/data/ALSF_pilot_data/SN0313537...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,/mnt/hdd20tb/alsf_ablation/SN0313537/BR0014397...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,xform_abl_gamma=155.18_699b97b2f9705fc2,albumentations:RandomGamma:699b97b2f9705fc2,"{""backend"":""albumentations"",""transform_name"":""...",[],...,-0.000646,0.000646,-0.000006,3.0,False,albumentations,RandomGamma,699b97b2f9705fc2,foreground_ssim,0.457415
2,20260219T025541654122Z,41811ddb-a68d-4909-830b-cee3d15b03ee,/mnt/data_nvme1/data/ALSF_pilot_data/SN0313537...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,/mnt/hdd20tb/alsf_ablation/SN0313537/BR0014397...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,xform_abl_gamma=193.32_0c77f398c28c2341,albumentations:RandomGamma:0c77f398c28c2341,"{""backend"":""albumentations"",""transform_name"":""...",[],...,-0.000646,0.000646,-0.000006,3.0,False,albumentations,RandomGamma,0c77f398c28c2341,ssim,0.714855
3,20260219T025541654122Z,41811ddb-a68d-4909-830b-cee3d15b03ee,/mnt/data_nvme1/data/ALSF_pilot_data/SN0313537...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,/mnt/hdd20tb/alsf_ablation/SN0313537/BR0014397...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,xform_abl_gamma=155.18_699b97b2f9705fc2,albumentations:RandomGamma:699b97b2f9705fc2,"{""backend"":""albumentations"",""transform_name"":""...",[],...,-0.000646,0.000646,-0.000006,3.0,False,albumentations,RandomGamma,699b97b2f9705fc2,mae,0.006020
4,20260218T212130127579Z,0f57f65f-729e-4cc4-a651-121c34b36181,/mnt/data_nvme1/data/ALSF_pilot_data/SN0313537...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,/mnt/hdd20tb/alsf_ablation/SN0313537/BR0014397...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,xform_abl_gaussnoise=0.1_e473bcd5b5024cf6,albumentations:GaussNoise:e473bcd5b5024cf6,"{""backend"":""albumentations"",""transform_name"":""...",[],...,-0.000646,0.000646,-0.000006,3.0,False,albumentations,GaussNoise,e473bcd5b5024cf6,foreground_ssim,0.143215


## Shared boostrap/regression parameters
All regression analysis will share the same dependent variable, whichare the metric values as well as the first (restricted) independent variable which will be the parameter value. The full independent variable and the groupings of regression analysis will change based on the confounding variable being tested for.

In [5]:
regression_config = {
    "y": "metric_value",    # dependent variable, always metric value for this analysis
    "x1": "param_values",   # independent variable 1, always the ablation parameter values for this analysis
}

bootstrap_config = {
    "n_boot": 300,
    "sample_frac": 0.5,
    "replace": True,
    "standardize": False,
    "robust_cov": None,     # or "HC3"
    "min_group_size": 25,   # prevent regression on tiny groups
}

## Regression Analysis 1: Assessing confounding by seeding density

In [6]:
colspec = ColumnSpec(
    group_cols=("metric_name", "ablation_type"),
    x2="seeding_density", # full regression parameters
    x2_categorical=False,
    standardize_cols=("param_values", "seeding_density"),
    **regression_config
)

cfg = BootstrapConfig(
    **bootstrap_config
)

boot_res = bootstrap_nested_regression(regression_input, colspec, cfg)
boot_res.to_parquet(results_dir / "boot_nest_confluence.parquet", index=False)

Bootstrap groups:   0%|          | 0/42 [00:00<?, ?it/s]

In [7]:
summarize_r2_scatter_bootstrap(
    boot_res,
    output_csv=results_dir / "boot_nest_confluence_summary.csv",
)

,metric_name,ablation_type,n_boot,restricted_r2_mean,restricted_r2_lower,restricted_r2_upper,partial_r2_mean,partial_r2_lower,partial_r2_upper
0,dists,Dilate,300,0.498193,0.490481,0.506279,0.110758,1.032898e-01,0.117899
1,dists,Erode,300,0.321256,0.302854,0.340736,0.080498,7.062043e-02,0.090313
2,dists,GaussNoise,300,0.174465,0.164639,0.183303,0.078161,7.204976e-02,0.084883
3,dists,GaussianBlur,300,0.072706,0.066758,0.078518,0.089581,8.414954e-02,0.095466
4,dists,GridDistortion,300,0.571806,0.563669,0.581536,0.055190,4.879735e-02,0.060850
5,dists,RandomGamma,300,0.357259,0.347118,0.367703,0.000294,4.798050e-05,0.000646
6,foreground_psnr,Dilate,300,0.107254,0.101385,0.112247,0.000264,5.710948e-05,0.000625
7,foreground_psnr,Erode,300,0.135684,0.127479,0.143064,0.029366,2.501001e-02,0.033490
8,foreground_psnr,GaussNoise,300,0.712901,0.705940,0.719638,0.059674,5.169054e-02,0.068869
9,foreground_psnr,GaussianBlur,300,0.178592,0.168815,0.187460,0.005385,3.696183e-03,0.007236


## Regression Analysis 2: Assessing confounding by cell lines

In [8]:
colspec = ColumnSpec(
    group_cols=("metric_name", "ablation_type"),
    x2="cell_line", # categorical var
    x2_categorical=True,
    standardize_cols=("param_values",),
    **regression_config
)

cfg = BootstrapConfig(
    **bootstrap_config
)

boot_res = bootstrap_nested_regression(regression_input, colspec, cfg)
boot_res.to_parquet(results_dir / "boot_nest_cell_line.parquet", index=False)

Bootstrap groups:   0%|          | 0/42 [00:00<?, ?it/s]

In [9]:
summarize_r2_scatter_bootstrap(
    boot_res,
    output_csv=results_dir / "boot_nest_cell_line_summary.csv",
)

,metric_name,ablation_type,n_boot,restricted_r2_mean,restricted_r2_lower,restricted_r2_upper,partial_r2_mean,partial_r2_lower,partial_r2_upper
0,dists,Dilate,300,0.498193,0.490481,0.506279,0.110159,0.102869,0.118532
1,dists,Erode,300,0.321256,0.302854,0.340736,0.126893,0.111868,0.141605
2,dists,GaussNoise,300,0.174465,0.164639,0.183303,0.028320,0.024309,0.032695
3,dists,GaussianBlur,300,0.072706,0.066758,0.078518,0.046145,0.040798,0.051156
4,dists,GridDistortion,300,0.571806,0.563669,0.581536,0.052346,0.047063,0.058327
5,dists,RandomGamma,300,0.357259,0.347118,0.367703,0.017781,0.015388,0.020288
6,foreground_psnr,Dilate,300,0.107254,0.101385,0.112247,0.304774,0.297129,0.312580
7,foreground_psnr,Erode,300,0.135684,0.127479,0.143064,0.133617,0.125048,0.141118
8,foreground_psnr,GaussNoise,300,0.712901,0.705940,0.719638,0.072444,0.062171,0.083076
9,foreground_psnr,GaussianBlur,300,0.178592,0.168815,0.187460,0.333650,0.321679,0.345457
